# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and extracts trades
of the **top 5% wallets** identified on the training period.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [1]:
import json
import datetime
from pathlib import Path

import pandas as pd

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

## Parameters

In [ ]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2026, 1, 1)
END_DATE   = datetime.date(2026, 1, 5)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-5% wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 1, 3)

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

We parse every `market_json` to build:
1. **token_lookup** – maps `token_id → {condition_id, outcome, token_winner, final_price}`
2. **market_meta** – per-market DataFrame with `condition_id`, `question`, `end_date`, `market_slug`

In [3]:
# Load only the market files whose month falls within [START_DATE, END_DATE].
# [:1] keeps execution fast; increase the slice to load more markets.
import datetime as _dt
def _file_in_range(p, start, end):
    """Return True if YYYY-MM.parquet filename falls within the date range."""
    try:
        y, m = (int(x) for x in p.stem.split("-"))
        file_date = _dt.date(y, m, 1)
        return start.replace(day=1) <= file_date <= end.replace(day=1)
    except Exception:
        return False

market_files = [
    p for p in sorted(MARKETS_DIR.glob("*.parquet"))
    if _file_in_range(p, START_DATE, END_DATE)
]
print(f"Found {len(market_files)} market file(s)")

all_market_rows = pd.concat(
    [pd.read_parquet(f) for f in market_files],
    ignore_index=True,
)
# deduplicate by condition_id (keep first occurrence)
all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
print(f"Unique markets (all):  {len(all_market_rows):,}")

# ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# Parse end_date_iso as a date; keep only markets whose resolution date falls
# within [START_DATE, END_DATE] (inclusive).  Markets outside this range are
# excluded, and their tokens will not appear in the token_lookup, so any trades
# on those tokens will be dropped during the enrichment join.
end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
all_market_rows = all_market_rows[in_range].reset_index(drop=True)
print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
all_market_rows.head(2)

Found 3 market file(s)


Unique markets (all):  303,101
Unique markets (filtered 2026-01-01 → 2026-03-20): 293,807


,end_date_iso,condition_id,market_json
0,2026-01-01T00:00:00Z,0x006ce0742c3891c396aead079e563c5d58c4eae2f9b0...,"{""enable_order_book"":false,""active"":true,""clos..."
1,2026-01-01T00:00:00Z,0x008add40355561724afbce56f68f1aa4ca4d83d8bdba...,"{""enable_order_book"":false,""active"":true,""clos..."


In [4]:
def build_lookups(market_rows: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Parse market_json column and return (token_lookup, market_meta_df)."""
    token_lookup: dict[str, dict] = {}
    meta_rows: list[dict] = []

    for _, row in market_rows.iterrows():
        try:
            m = json_loads(row["market_json"])
        except Exception:
            continue

        cid = str(m.get("condition_id", row["condition_id"]))

        # ── token resolution lookup ───────────────────────────────────────────
        for tok in m.get("tokens", []):
            token_id = str(tok["token_id"])
            winner = bool(tok.get("winner", False))
            # final_price is determined solely by the token winner flag:
            # winner=True  → settled at 1.0 USDC per share
            # winner=False → settled at 0.0 USDC per share
            final_price = 1.0 if winner else 0.0
            token_lookup[token_id] = {
                "condition_id": cid,
                "outcome":      tok.get("outcome", ""),
                "token_winner": winner,
                "final_price":  final_price,
            }

        # ── market metadata ───────────────────────────────────────────────────
        meta_rows.append({
            "condition_id": cid,
            "question":     m.get("question", ""),
            "end_date":     pd.to_datetime(m.get("end_date_iso"), utc=True, errors="coerce"),
            "market_slug":  m.get("market_slug", ""),
        })

    market_meta = pd.DataFrame(meta_rows)
    return token_lookup, market_meta


token_lookup, market_meta = build_lookups(all_market_rows)
print(f"Token lookup entries: {len(token_lookup):,}")
print(f"Market meta rows:     {len(market_meta):,}")

Token lookup entries: 583,607
Market meta rows:     293,807


## 2 · Process partitioned per-side trades and filter to in-range markets

Trades are already stored as wallet-perspective rows (`wallet`, `side`) and
`TRADES_DIR` is partitioned, so we process each parquet shard independently to
avoid loading all fills into memory at once. For each shard we:
1. Inner-join with token resolution (`token_winner`, `final_price`)
2. Build `final_value_usdc = quantity × final_price`
3. Aggregate to wallet × market × timestamp and accumulate global results


In [5]:
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
if not trade_files:
    raise FileNotFoundError(f"No parquet files found in {TRADES_DIR}")

print(f"Found {len(trade_files)} trade parquet file(s)")

sample_raw = pd.read_parquet(trade_files)
print(f"Sample shard rows ({trade_files[0].name}): {len(sample_raw):,}")
print(f"Columns: {sample_raw.columns.tolist()}")
if "trade_date" in sample_raw.columns:
    print(f"Sample shard date range: {sample_raw['trade_date'].min()} → {sample_raw['trade_date'].max()}")
sample_raw.head(3)


Found 16 trade parquet file(s)


Sample shard rows (0.parquet): 93,357,019
Columns: ['tx_hash', 'log_index', 'block_number', 'block_timestamp', 'trade_date', 'condition_id', 'token_id', 'outcome', 'wallet', 'side', 'price', 'quantity', 'usdc_amount', 'question', 'slug', 'fill_count', 'position', 'flipped', 'split', 'tags', 'raw_side', 'raw_outcome', 'raw_token_id', 'raw_price', 'raw_usdc_amount']


Sample shard date range: 2025-01-01 → 2026-03-11


,tx_hash,log_index,block_number,block_timestamp,trade_date,condition_id,token_id,outcome,wallet,side,...,fill_count,position,flipped,split,tags,raw_side,raw_outcome,raw_token_id,raw_price,raw_usdc_amount
0,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e...,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,7875320516565813053445635107732149656386289192...,No,0x126f15c7bd21bf5bf136f3629e10990dc0677137,BUY,...,1,10.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 20...",BUY,No,7875320516565813053445635107732149656386289192...,0.62,6.2
1,0xdfa6affbb564cebba0f93ad102af9f5b9971270f1c9e...,936,66158948,1735689665,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,5866505527798653441671993940591462145801013733...,Yes,0x3978a99e028eb590c98d8e9ddbe513298fa92f24,BUY,...,1,10.0,True,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 20...",SELL,No,7875320516565813053445635107732149656386289192...,0.62,6.2
2,0xc12324352ff1897431a9cdd5e91bcc05cc7c28e88c38...,1531,66159089,1735689997,2025-01-01,0x0f416235a6d63a19f2779906242ce173aec3e49bbdcd...,5866505527798653441671993940591462145801013733...,Yes,0x260d1ae03c985f7c78ad286b5da14940b4739679,BUY,...,1,30.0,False,False,"[Science, Culture, H5N1, vaccine, Bird Flu, 20...",BUY,Yes,5866505527798653441671993940591462145801013733...,0.37,11.1


In [6]:
# ── build token resolution DataFrame from lookup ─────────────────────────────
token_df = pd.DataFrame.from_dict(token_lookup, orient="index")
token_df.index.name = "token_id"
token_df.reset_index(inplace=True)
token_df["token_id"] = token_df["token_id"].astype(str)

def enrich_trade_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    """Attach settlement columns and computed datetime/value to one shard."""
    if chunk.empty:
        return chunk

    c = chunk.copy()
    c["token_id"] = c["token_id"].astype(str)
    c = c.merge(
        token_df[["token_id", "token_winner", "final_price"]],
        on="token_id",
        how="inner",
    )
    if c.empty:
        return c

    c["dt"] = pd.to_datetime(c["block_timestamp"], unit="s", utc=True)
    c["final_value_usdc"] = c["quantity"] * c["final_price"]
    return c

GROUP_KEYS = ["wallet", "condition_id", "dt"]

READ_COLS = [
    "tx_hash", "log_index", "block_timestamp", "trade_date", "condition_id",
    "token_id", "outcome", "price", "quantity", "usdc_amount", "position",
    "wallet", "side",
]

grouped_parts: list[pd.DataFrame] = []
sample_fills = None

total_raw_rows = 0
total_filtered_rows = 0

for i, f in enumerate(trade_files, start=1):
    raw_chunk = pd.read_parquet(f, columns=READ_COLS)
    total_raw_rows += len(raw_chunk)

    enriched_chunk = enrich_trade_chunk(raw_chunk)
    total_filtered_rows += len(enriched_chunk)

    if sample_fills is None and len(enriched_chunk) > 0:
        sample_fills = enriched_chunk.head(5).copy()

    if len(enriched_chunk) == 0:
        continue

    grouped_part = (
        enriched_chunk.groupby(GROUP_KEYS, sort=False)
        .agg(
            side             = ("side",             "first"),
            outcome          = ("outcome",          "first"),
            token_winner     = ("token_winner",     "first"),
            final_price      = ("final_price",      "first"),
            position         = ("position",         "max"),
            total_quantity   = ("quantity",         "sum"),
            price_sum        = ("price",            "sum"),
            price_count      = ("price",            "count"),
            trade_value_usdc = ("usdc_amount",      "sum"),
            final_value_usdc = ("final_value_usdc", "sum"),
            num_fills        = ("tx_hash",          "count"),
        )
        .reset_index()
    )
    grouped_parts.append(grouped_part)

    if i % 25 == 0 or i == len(trade_files):
        print(
            f"Processed {i}/{len(trade_files)} shards | "
            f"raw rows: {total_raw_rows:,} | in-range rows: {total_filtered_rows:,}"
        )

if not grouped_parts:
    raise ValueError("No in-range trade rows after token filter")

grouped = pd.concat(grouped_parts, ignore_index=True)

# In case a key spans multiple shards, combine partial aggregates.
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        side             = ("side",             "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_sum        = ("price_sum",        "sum"),
        price_count      = ("price_count",      "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
    )
    .reset_index()
)

grouped["avg_price"] = grouped["price_sum"] / grouped["price_count"]
grouped.drop(columns=["price_sum", "price_count"], inplace=True)
grouped.sort_values(["wallet", "condition_id", "dt"], inplace=True, ignore_index=True)

print(
    f"Rows after market filter: {total_filtered_rows:,}  "
    f"(dropped {total_raw_rows - total_filtered_rows:,} rows outside [{START_DATE}, {END_DATE}])"
)
print(f"Prepared grouped rows: {len(grouped):,}")
if sample_fills is not None:
    sample_fills.head(5)


Processed 16/16 shards | raw rows: 93,357,019 | in-range rows: 57,193,947


Rows after market filter: 57,193,947  (dropped 36,163,072 rows outside [2026-01-01, 2026-03-20])
Prepared grouped rows: 40,187,923


## 3 · Enriched fill table

All fills are already filtered to in-range markets and carry resolution columns
from the inner join performed in the previous step.

| column | description |
|---|---|
| `token_winner` | `True` if the traded token is the winning outcome |
| `final_price` | Settlement price: 1.0 if `token_winner=True`, 0.0 otherwise |
| `final_value_usdc` | `quantity × final_price` — USDC value of the position at settlement |

In [7]:
if sample_fills is None:
    print("No enriched fills available after filtering")
    pd.DataFrame(columns=[
        "wallet", "side", "condition_id", "dt", "outcome", "quantity", "price",
        "position", "usdc_amount", "token_winner", "final_price", "final_value_usdc"
    ])
else:
    print(f"Raw rows scanned across all shards: {total_raw_rows:,}")
    print(f"In-range rows after token join:     {total_filtered_rows:,}")
    print(f"token_winner null: {sample_fills['token_winner'].isna().sum():,}  (sample should be 0)")
    sample_fills[["wallet", "side", "condition_id", "dt", "outcome", "quantity", "price", "position",
                  "usdc_amount", "token_winner", "final_price", "final_value_usdc"]].head(5)


Raw rows scanned across all shards: 93,357,019
In-range rows after token join:     57,193,947
token_winner null: 0  (sample should be 0)


## 4 · Group by wallet × market × timestamp

A single on-chain transaction can generate multiple fills at the same timestamp.
Each row in `grouped` aggregates all fills for one wallet in one market at one timestamp.

In [8]:
print(f"Grouped rows: {len(grouped):,}")
grouped.head(10)


Grouped rows: 40,187,923


,wallet,condition_id,dt,side,outcome,token_winner,final_price,position,total_quantity,trade_value_usdc,final_value_usdc,num_fills,avg_price
0,0x0000000433698c8f7f08eb24c57f1d62a6fd946e,0x4b78de44ea49d0eb8d5fee85352294b6f679a47c5694...,2026-02-14 01:47:41+00:00,BUY,No,True,1.0,1.184833,1.184833,0.999999,1.184833,1,0.844
1,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x2ece3af7241e819906ef9f99cc850ba7feaa7f7dc41a...,2026-01-09 07:47:14+00:00,BUY,No,True,1.0,13.052000,13.052000,12.999792,13.052000,1,0.996
2,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x79d7663a5a4b32512cbb86ce4b1a465d0bee5ec544e2...,2026-02-10 08:03:07+00:00,BUY,Yes,True,1.0,0.700000,0.700000,0.693000,0.700000,1,0.990
3,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x79d7663a5a4b32512cbb86ce4b1a465d0bee5ec544e2...,2026-02-10 08:03:35+00:00,BUY,Yes,True,1.0,15.900000,15.200000,15.063200,15.200000,1,0.991
4,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x03e38f4c5abeb1560d3af133186dd3130f8018336496...,2026-03-03 01:25:43+00:00,BUY,Clippers,True,1.0,10.000000,10.000000,5.100000,10.000000,1,0.510
5,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x0cb8af5c2d8b2c03561f509af4fcd1f592f4c962b20e...,2026-02-06 15:14:27+00:00,BUY,Wizards,False,0.0,5.000000,5.000000,2.050000,0.000000,1,0.410
6,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x174bb97123584e2a0077b82660b0252e37d7ec347f1c...,2026-02-13 22:18:49+00:00,BUY,Ireland,True,1.0,5.000000,5.000000,3.800000,5.000000,1,0.760
7,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x19a68fc8ed914fb4b33007cb2ab49ebffec3fdc6a0f4...,2026-03-05 10:47:21+00:00,BUY,Under,False,0.0,4.895832,4.895832,2.349999,0.000000,1,0.480
8,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x1eff235952b261d4f6dd116f489c90b4172da64bbad1...,2026-01-14 11:03:20+00:00,BUY,Blackhawks,False,0.0,5.000000,5.000000,2.600000,0.000000,1,0.520
9,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x1feb9dfc82cba526c2c79fcc2e5fde7e7c03f78e0960...,2026-03-01 21:42:17+00:00,BUY,Grizzlies,True,1.0,10.000000,10.000000,5.000000,10.000000,1,0.500


## 5 · Per-trade P&L and wallet summary (training data only)

P&L is calculated **per trade** based on whether the traded token is a winner:

| side | `trade_pnl` formula | intuition |
|---|---|---|
| BUY | `final_value_usdc − trade_value_usdc` | bought tokens; profit if `final_price > entry_price` |
| SELL | `trade_value_usdc − final_value_usdc` | sold tokens; profit if sold above final settlement price |

where `final_value_usdc = total_quantity × final_price` (1.0 if the token is a winner, 0.0 otherwise).

Wallet P&L = `Σ trade_pnl` over training trades.

In [9]:
# ── per-trade P&L: final_value vs. trade_value, signed by side ──────────────
# BUY:  wallet paid trade_value_usdc, ends up with tokens worth final_value_usdc
#       → pnl = final_value_usdc - trade_value_usdc  (positive if token wins)
# SELL: wallet received trade_value_usdc, gave up tokens worth final_value_usdc
#       → pnl = trade_value_usdc - final_value_usdc  (positive if token loses)
is_buy = grouped["side"] == "BUY"
grouped["trade_pnl"] = (
    (grouped["final_value_usdc"] - grouped["trade_value_usdc"]).where(is_buy,
    grouped["trade_value_usdc"] - grouped["final_value_usdc"])
)

# ── restrict to training period for wallet ranking ────────────────────────────
end_train_ts   = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train  = grouped[grouped["dt"] < end_train_ts]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets      = ("condition_id",   "nunique"),
        num_trades       = ("num_fills",       "sum"),
        total_cost_usdc  = ("trade_value_usdc", "sum"),
        pnl_usdc         = ("trade_pnl",       "sum")
                )
    .sort_values("pnl_usdc", ascending=False)
    .reset_index()
)

print(f"Unique wallets (training period): {len(wallet_summary):,}")
wallet_summary.head(5)

Unique wallets (training period): 394,972


,wallet,num_markets,num_trades,total_cost_usdc,pnl_usdc
0,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,663,52672,4.946852e+07,6.716902e+06
1,0xdb27bf2ac5d428a9c63dbc914611036855a6c56e,319,50996,2.666777e+07,2.908843e+06
2,0x876426b52898c295848f56760dd24b55eda2604a,76,10113,3.927546e+06,1.554148e+06
3,0x1bc0d88ca86b9049cf05d642e634836d5ddf4429,183,14234,7.482051e+06,1.482843e+06
4,0x91654fd592ea5339fc0b1b2f2b30bfffa5e75b98,115,5752,1.060784e+07,1.386730e+06


## 6 · Market-level volume summary

In [10]:
# join market metadata (question, end_date) for display
grouped_meta = grouped.merge(
    market_meta[["condition_id", "question", "end_date", "market_slug"]],
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum")
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 142,290


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3...,US government shutdown Saturday?,2026-01-31 00:00:00+00:00,27577,351968,1.394019e+08,1.416030e+08
1,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,Khamenei out as Supreme Leader of Iran by Febr...,2026-02-28 00:00:00+00:00,22553,280093,1.089277e+08,1.182862e+08
2,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,"US strikes Iran by February 28, 2026?",2026-01-31 00:00:00+00:00,22906,268526,7.830458e+07,9.590727e+07
3,0xa8b744720006da3c08b4dc8a61a5ce930542f550fcf8...,Khamenei out as Supreme Leader of Iran by Janu...,2026-01-31 00:00:00+00:00,21688,182262,4.427491e+07,4.359977e+07
4,0xabb86b080e9858dcb3f46954010e49b6f539c2003685...,"US strikes Iran by January 31, 2026?",2026-01-31 00:00:00+00:00,21573,258175,3.829641e+07,3.778333e+07
5,0x204d24f3a0f5dd5fca825292bdeab6a97af3978b2caa...,Seahawks vs. Patriots,2026-02-08 00:00:00+00:00,16979,124904,3.472805e+07,3.355085e+07
6,0x05beef793157deb1cc34e2d3159539f461b73c7eeaa3...,U.S. anti-cartel ground operation in Mexico by...,2026-01-31 00:00:00+00:00,3356,36195,2.792410e+07,2.799013e+07
7,0xb3dbb0f2e1df15cc820e0cde749eb112bf160994ec53...,"Will Melania say ""Career"" during AI talk on Fr...",2026-01-16 00:00:00+00:00,2520,14066,2.788430e+07,2.786857e+07
8,0xb8c1bd306a8a4cedfb280e114e655c5092b3f37edcca...,"Russia x Ukraine ceasefire by January 31, 2026?",2026-01-31 00:00:00+00:00,12429,102342,2.418948e+07,2.393021e+07
9,0x09cbe3e796661a1d820580145488ad2ccb9ad1e720ef...,"US strikes Iran by February 27, 2026?",2026-01-31 00:00:00+00:00,8845,96213,2.206237e+07,2.190368e+07


## 7 · Wallet statistics (quantiles)

Distribution of activity and P&L across wallets (training period).

In [11]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "pnl_usdc"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

# number of wallets at or below each quantile threshold
wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])

# append count, mean and std for reference
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")

wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]   = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"]  = wallet_stats["num_markets"].round(1)
wallet_stats["pnl_usdc"]     = wallet_stats["pnl_usdc"].round(2)

wallet_stats

,wallet_count,num_trades,num_markets,pnl_usdc,wallet_count_complement
quantile,,,,,
0.001,395,1.0,1.0,-33278.58,394577
0.01,3950,1.0,1.0,-3657.98,391022
0.05,19749,1.0,1.0,-694.88,375223
0.1,39497,1.0,1.0,-183.50,355475
0.25,98743,2.0,1.0,-23.44,296229
0.5,197486,4.0,2.0,-0.78,197486
0.75,296229,15.0,6.0,3.26,98743
0.9,355475,59.0,19.0,72.14,39497
0.95,375223,155.0,45.0,387.60,19749


## 8 · Full enriched trade table

One row per wallet × market × timestamp, with all enrichment columns.

In [12]:
DISPLAY_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,wallet,condition_id,dt,side,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,num_fills
0,0x0000000433698c8f7f08eb24c57f1d62a6fd946e,0x4b78de44ea49d0eb8d5fee85352294b6f679a47c5694...,2026-02-14 01:47:41+00:00,BUY,No,1.184833,1.184833,0.844,True,1.0,0.999999,1.184833,0.184834,1
1,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x2ece3af7241e819906ef9f99cc850ba7feaa7f7dc41a...,2026-01-09 07:47:14+00:00,BUY,No,13.052000,13.052000,0.996,True,1.0,12.999792,13.052000,0.052208,1
2,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x79d7663a5a4b32512cbb86ce4b1a465d0bee5ec544e2...,2026-02-10 08:03:07+00:00,BUY,Yes,0.700000,0.700000,0.990,True,1.0,0.693000,0.700000,0.007000,1
3,0x0000075afc60ebc564d56be0bc23eaceea67d446,0x79d7663a5a4b32512cbb86ce4b1a465d0bee5ec544e2...,2026-02-10 08:03:35+00:00,BUY,Yes,15.900000,15.200000,0.991,True,1.0,15.063200,15.200000,0.136800,1
4,0x000008f5286b28d9447c488fe3c6ee9bac4e9d64,0x03e38f4c5abeb1560d3af133186dd3130f8018336496...,2026-03-03 01:25:43+00:00,BUY,Clippers,10.000000,10.000000,0.510,True,1.0,5.100000,10.000000,4.900000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40187918,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,2026-02-05 16:37:39+00:00,BUY,Yes,150.000000,150.000000,0.080,True,1.0,12.000000,150.000000,138.000000,3
40187919,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,2026-02-05 17:13:43+00:00,BUY,Yes,350.000000,200.000000,0.080,True,1.0,16.000000,200.000000,184.000000,1
40187920,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,2026-02-05 22:23:27+00:00,BUY,Yes,550.000000,200.000000,0.070,True,1.0,14.000000,200.000000,186.000000,1
40187921,0xffffffe1e093aacd21e4e281e66d543fb0b23455,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,2026-02-28 16:39:01+00:00,SELL,Yes,530.000000,50.000000,0.507,True,1.0,25.350000,50.000000,-24.650000,2


## 9 · Export top-5% wallets to parquet shards

Identifies top-5% wallets by P&L on the **training period**, then performs a
second partition-by-partition pass to export only those wallets' trades from the
**full dataset** (training + test) to `data/polygon_trades_processed/*.parquet`.

Each row is a single per-side fill, enriched with `final_value_usdc` and a boolean
`is_train` flag. Output remains partitioned by input shard.


In [13]:
# ── identify top-5% wallets by training-period P&L ───────────────────────────
pnl_threshold = wallet_summary["pnl_usdc"].quantile(0.95)
top_wallets   = set(wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "wallet"])
print(f"P&L threshold (95th pct, training): {pnl_threshold:,.2f} USDC")
print(f"Top-5% wallet count:                {len(top_wallets):,}")

# print deciles of training P&L for top wallets
top_pnl = wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "pnl_usdc"]
for i in range(0, 11):
    q = i / 10
    print(f"  P&L at {q:.0%} pct of top wallets: {top_pnl.quantile(q):,.2f} USDC")

P&L threshold (95th pct, training): 387.60 USDC
Top-5% wallet count:                19,749
  P&L at 0% pct of top wallets: 387.62 USDC
  P&L at 10% pct of top wallets: 491.78 USDC
  P&L at 20% pct of top wallets: 622.76 USDC
  P&L at 30% pct of top wallets: 745.00 USDC
  P&L at 40% pct of top wallets: 1,000.02 USDC
  P&L at 50% pct of top wallets: 1,336.71 USDC
  P&L at 60% pct of top wallets: 1,694.64 USDC
  P&L at 70% pct of top wallets: 2,174.40 USDC
  P&L at 80% pct of top wallets: 3,120.62 USDC
  P&L at 90% pct of top wallets: 6,059.91 USDC
  P&L at 100% pct of top wallets: 6,716,901.62 USDC


In [14]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "token_id", "outcome",
    "quantity", "price", "usdc_amount", "position",
    "final_value_usdc", "trade_pnl",
    "token_winner", "final_price",
    "tx_hash",
    "is_train",
]

EXPORT_READ_COLS = [
    "tx_hash", "log_index", "block_timestamp", "condition_id", "token_id",
    "outcome", "price", "quantity", "usdc_amount", "position", "wallet", "side",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

total_top_rows = 0
total_top_train_rows = 0
top_trades_preview = None
output_files = []

for i, f in enumerate(trade_files, start=1):
    raw_chunk = pd.read_parquet(f, columns=EXPORT_READ_COLS)
    enriched_chunk = enrich_trade_chunk(raw_chunk)
    if len(enriched_chunk) == 0:
        continue

    top_chunk = enriched_chunk[enriched_chunk["wallet"].isin(top_wallets)].copy()
    if len(top_chunk) == 0:
        continue

    is_buy_fill = top_chunk["side"] == "BUY"
    top_chunk["trade_pnl"] = (
        (top_chunk["final_value_usdc"] - top_chunk["usdc_amount"]).where(
            is_buy_fill,
            top_chunk["usdc_amount"] - top_chunk["final_value_usdc"],
        )
    )
    top_chunk["is_train"] = top_chunk["dt"].dt.date <= END_DATE_TRAIN

    export_chunk = top_chunk[EXPORT_COLS]

    out_file = OUT_DIR / f.name
    export_chunk.to_parquet(out_file, index=False)
    output_files.append(out_file)

    if top_trades_preview is None:
        top_trades_preview = export_chunk.head(5).copy()

    total_top_rows += len(export_chunk)
    total_top_train_rows += int(export_chunk["is_train"].sum())

    if i % 25 == 0 or i == len(trade_files):
        print(
            f"Processed export shard {i}/{len(trade_files)} | "
            f"accumulated rows: {total_top_rows:,} | output shards: {len(output_files):,}"
        )

if not output_files:
    empty_file = OUT_DIR / "part-00000.parquet"
    pd.DataFrame(columns=EXPORT_COLS).to_parquet(empty_file, index=False)
    output_files = [empty_file]

top_trades_count = total_top_rows

print(f"Total expanded rows for top wallets: {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
print(f"\nSaved partitioned output → {OUT_DIR}")


Processed export shard 16/16 | accumulated rows: 20,808,775 | output shards: 16
Total expanded rows for top wallets: 20,808,775
  training rows: 14,053,676
  test rows:     6,755,099
Output shards:  16

Saved partitioned output → ../../data/polygon_trades_processed


In [15]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview


### Unit test: CTF Exchange contract is not treated as a trader

`0x85d355ef32d62b09d2362184f299a7fc662ee1a4` is the Polymarket CTF Exchange
(on-chain matching contract). It can appear in per-side trade rows as a `wallet`
counterparty, but it is **not** a real trader.

This test always enforces that the exchange address is **not** in `top_wallets`.
It also reports whether the exchange appears in the currently filtered dataset
(appearance depends on the chosen date range).


In [16]:
# ── Unit test: CTF Exchange contract is not selected as a top wallet ──────────
CTF_EXCHANGE = "0x85d355ef32d62b09d2362184f299a7fc662ee1a4"

# Presence in grouped depends on date filtering; this is informational.
exchange_rows = grouped[grouped["wallet"] == CTF_EXCHANGE]
if len(exchange_rows) > 0:
    print(f"INFO  CTF Exchange appears in grouped:  {len(exchange_rows):,} rows")
else:
    print("INFO  CTF Exchange rows not present in current filtered dataset")

# Must always hold regardless of date range.
assert CTF_EXCHANGE not in top_wallets, (
    f"CTF Exchange contract ({CTF_EXCHANGE}) was incorrectly selected as a top wallet. "
    "It is the on-chain matching engine, not a real trader."
)

print(f"PASS  CTF Exchange not in top_wallets: top_wallets has {len(top_wallets):,} entries")


INFO  CTF Exchange appears in grouped:  4 rows
PASS  CTF Exchange not in top_wallets: top_wallets has 19,749 entries
